# FinBERT Full-Data Staged Workflow

Colab-first staged workflow for LM2011-scale FinBERT processing. Sentence parquet is the only v1 handoff artifact between stages: sentence preprocessing, tokenizer profiling, and model inference.

In [6]:
from pathlib import Path

# Colab bootstrap / install
from google.colab import drive

drive.mount("/content/drive")
#REPO_ROOT = Path("/content/drive/MyDrive/Data_LM/code/NLP_Thesis")
# %cd {REPO_ROOT}
# %pip install -e .[benchmark]

print("If this is a fresh runtime, restart the runtime after the first install.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
If this is a fresh runtime, restart the runtime after the first install.


In [7]:
from pathlib import Path

REPO_ROOT = Path("/content/NLP_Thesis")
%cd /content
!rm -rf {REPO_ROOT}
!git clone https://github.com/Erew42/NLP_Thesis.git {REPO_ROOT}
%cd {REPO_ROOT}
!git checkout f584764f29304cf5aee50bbeca25074c8d7a9ef0  # or your branch name
%pip install -e .[benchmark]


/content
Cloning into '/content/NLP_Thesis'...
remote: Enumerating objects: 1608, done.
remote: Counting objects: 100% (403/403), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 1608 (delta 268), reused 263 (delta 137), pack-reused 1205 (from 1)
Receiving objects: 100% (1608/1608), 1.82 MiB | 11.64 MiB/s, done.
Resolving deltas: 100% (1009/1009), done.
/content/NLP_Thesis
Note: switching to 'f584764f29304cf5aee50bbeca25074c8d7a9ef0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at f584764

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/NLP_Thesis")
sys.path.insert(0, str(REPO_ROOT / "src"))

import thesis_pkg
import thesis_pkg.benchmarking as b

print(thesis_pkg.__file__)
print(b.FinbertSentenceParquetInferenceRunConfig)


/content/NLP_Thesis/src/thesis_pkg/__init__.py
<class 'thesis_pkg.benchmarking.contracts.FinbertSentenceParquetInferenceRunConfig'>


In [8]:
from pathlib import Path

FULL_DATA_ROOT = Path("/content/drive/MyDrive/Data_LM")
ITEMS_ANALYSIS_DIR = FULL_DATA_ROOT / "results" / "sec_ccm_unified_runner" / "items_analysis"
BACKBONE_PATH = FULL_DATA_ROOT / "results" / "sec_ccm_unified_runner" / "sec_ccm_premerge" / "final_flagged_data.parquet"
SENTENCE_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_sentence_preprocessing"
TOKENIZER_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_tokenizer_profile"
MODEL_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_sentence_parquet_inference"

YEAR_FILTER = None  # Set to (2006, 2007, 2008) for smoke runs.
RUN_SENTENCE_STAGE = True
RUN_TOKENIZER_STAGE = True
RUN_MODEL_STAGE = False
WRITE_SENTENCE_SCORES = False


In [3]:
import polars as pl

from thesis_pkg.benchmarking import BucketBatchConfig
from thesis_pkg.benchmarking import FinbertRuntimeConfig
from thesis_pkg.benchmarking import FinbertSectionUniverseConfig
from thesis_pkg.benchmarking import FinbertSentenceParquetInferenceRunConfig
from thesis_pkg.benchmarking import FinbertSentencePreprocessingRunConfig
from thesis_pkg.benchmarking import FinbertTokenizerProfileRunConfig
from thesis_pkg.benchmarking import SentenceDatasetConfig
from thesis_pkg.benchmarking import run_finbert_sentence_parquet_inference
from thesis_pkg.benchmarking import run_finbert_sentence_preprocessing
from thesis_pkg.benchmarking import run_finbert_tokenizer_profile


In [4]:
SECTION_UNIVERSE = FinbertSectionUniverseConfig(source_items_dir=ITEMS_ANALYSIS_DIR)
SENTENCE_DATASET_CFG = SentenceDatasetConfig(enabled=True, spacy_batch_size=32, token_length_batch_size=1024)
TOKENIZER_BATCH_CFG = BucketBatchConfig(name="baseline", short_batch_size=64, medium_batch_size=32, long_batch_size=16)
MODEL_BATCH_CFG = BucketBatchConfig(name="baseline", short_batch_size=64, medium_batch_size=32, long_batch_size=16)
RUNTIME_CFG = FinbertRuntimeConfig(device=None, use_autocast=True, amp_dtype="auto")

SENTENCE_STAGE_RUN_NAME = "finbert_full_data_sentence_stage"
TOKENIZER_STAGE_RUN_NAME = "finbert_full_data_tokenizer_profile"
MODEL_STAGE_RUN_NAME = "finbert_full_data_model_inference"
SENTENCE_DATASET_DIR = SENTENCE_STAGE_OUT_ROOT / SENTENCE_STAGE_RUN_NAME / "sentence_dataset" / "by_year"


In [9]:
sentence_stage_artifacts = None
if RUN_SENTENCE_STAGE:
    sentence_stage_artifacts = run_finbert_sentence_preprocessing(
        FinbertSentencePreprocessingRunConfig(
            source_items_dir=ITEMS_ANALYSIS_DIR,
            out_root=SENTENCE_STAGE_OUT_ROOT,
            section_universe=SECTION_UNIVERSE,
            sentence_dataset=SENTENCE_DATASET_CFG,
            target_doc_universe_path=BACKBONE_PATH,
            year_filter=YEAR_FILTER,
            run_name=SENTENCE_STAGE_RUN_NAME,
        )
    )
    print(f"sentence_dataset_dir={sentence_stage_artifacts.sentence_dataset_dir}")
else:
    print("Sentence stage disabled.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

ValueError: [E088] Text of length 1032648 exceeds maximum of 1000000. The parser and NER models require roughly 1GB of temporary memory per 100,000 characters in the input. This means long texts may cause memory allocation errors. If you're not using the parser or NER, it's probably safe to increase the `nlp.max_length` limit. The limit is in number of characters, so you can check whether your inputs are too long by checking `len(text)`.

In [6]:
sentence_summary_path = SENTENCE_STAGE_OUT_ROOT / SENTENCE_STAGE_RUN_NAME / "sentence_dataset_yearly_summary.parquet"
if sentence_summary_path.exists():
    pl.read_parquet(sentence_summary_path)
else:
    print("Sentence stage artifacts not found yet.")


Sentence stage artifacts not found yet.


In [ ]:
tokenizer_stage_artifacts = None
if RUN_TOKENIZER_STAGE:
    tokenizer_stage_artifacts = run_finbert_tokenizer_profile(
        FinbertTokenizerProfileRunConfig(
            sentence_dataset_dir=SENTENCE_DATASET_DIR,
            out_root=TOKENIZER_STAGE_OUT_ROOT,
            batch_config=TOKENIZER_BATCH_CFG,
            year_filter=YEAR_FILTER,
            profile_row_cap_per_bucket=5000,
            run_name=TOKENIZER_STAGE_RUN_NAME,
        )
    )
    print(f"timing_summary={tokenizer_stage_artifacts.timing_summary_path}")
else:
    print("Tokenizer stage disabled.")


In [ ]:
tokenizer_summary_path = TOKENIZER_STAGE_OUT_ROOT / TOKENIZER_STAGE_RUN_NAME / "tokenizer_profile_bucket_summary.parquet"
if tokenizer_summary_path.exists():
    pl.read_parquet(tokenizer_summary_path)
else:
    print("Tokenizer profile artifacts not found yet.")


In [ ]:
model_stage_artifacts = None
if RUN_MODEL_STAGE:
    model_stage_artifacts = run_finbert_sentence_parquet_inference(
        FinbertSentenceParquetInferenceRunConfig(
            sentence_dataset_dir=SENTENCE_DATASET_DIR,
            out_root=MODEL_STAGE_OUT_ROOT,
            batch_config=MODEL_BATCH_CFG,
            runtime=RUNTIME_CFG,
            backbone_path=BACKBONE_PATH,
            year_filter=YEAR_FILTER,
            sentence_slice_rows=5000,
            write_sentence_scores=WRITE_SENTENCE_SCORES,
            run_name=MODEL_STAGE_RUN_NAME,
        )
    )
    print(f"item_features_long={model_stage_artifacts.item_features_long_path}")
else:
    print("Model stage disabled.")


In [ ]:
model_summary_path = MODEL_STAGE_OUT_ROOT / MODEL_STAGE_RUN_NAME / "model_inference_yearly_summary.parquet"
item_features_long_path = MODEL_STAGE_OUT_ROOT / MODEL_STAGE_RUN_NAME / "item_features_long.parquet"
if model_summary_path.exists():
    pl.read_parquet(model_summary_path)
elif item_features_long_path.exists():
    pl.read_parquet(item_features_long_path)
else:
    print("Model stage artifacts not found yet.")
